# PICK: Semi-Supervised Multi-Class Segmentation (BHSD Fold 0) với Weights & Biases (W&B)

Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background trên GPU Kaggle.

### Tính năng mới - Theo dõi tiến trình qua Weights & Biases:
- **Theo dõi thời gian thực (Real-time Cloud Tracking)**: Loss tổng thể, Main loss, MIM loss, Aux loss, Learning rate trực tiếp trên Cloud.
- **Biểu đồ mDice & Per-class Dice**: Cập nhật mỗi 200 bước kiểm định (Validation), hiển thị rõ nét trên web và điện thoại.
- **Theo dõi phần cứng tự động**: Tự động ghi nhận GPU utilization, VRAM usage, nhiệt độ GPU trên Kaggle.
- **Không phụ thuộc vào tab trình duyệt**: Bạn có thể tắt máy hoặc đóng tab Kaggle mà vẫn xem được tiến trình train qua mobile/laptop.

### Cấu hình Kaggle Notebook:
- **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100** (16GB VRAM).
- **Internet**: Bật **ON** (bắt buộc để đồng bộ lên W&B và clone code).
- **Dataset**: Bấm **+ Add Input** -> Chọn dataset `BHSD Fold 0` đã upload.

### Bước 1: Cài đặt các thư viện cần thiết
Cài đặt `medpy`, `tensorboardX`, `wandb` (Weights & Biases) cùng các thư viện xử lý ảnh y tế.

In [ ]:
!pip install -q medpy tensorboardX h5py SimpleITK nibabel wandb

### Bước 2: Clone & Cập nhật mã nguồn PICK từ GitHub

In [ ]:
import os

REPO_NAME = "NguyenVanCongThanh/PICK_BHSD"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

WORK_DIR = "/kaggle/working/PICK"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} {WORK_DIR}
else:
    print(f"[OK] Cập nhật git pull tại {WORK_DIR}...")
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!git log -1 --oneline

### Bước 3: Định vị Dataset BHSD Fold 0 từ Kaggle Input

In [ ]:
import os, glob

INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

def find_bhsd_root(base_dir):
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        if "data" in dirs and ("splits" in dirs or "fold0_train.txt" in files):
            return root
    return None

print("[*] Đang định vị dataset BHSD Fold 0...")
DATA_ROOT = find_bhsd_root(INPUT_DIR) or find_bhsd_root(WORKING_DIR)

if not DATA_ROOT:
    all_tars = glob.glob(os.path.join(INPUT_DIR, "**", "*.tar"), recursive=True)
    matching_tars = [p for p in all_tars if "bhsd" in os.path.basename(p).lower() or "fold0" in os.path.basename(p).lower()]
    tar_path = matching_tars[0] if matching_tars else (all_tars[0] if all_tars else None)
    
    if not tar_path:
        raise FileNotFoundError(f"[X] Không tìm thấy dataset BHSD Fold 0 trong {INPUT_DIR}! Hãy bấm '+ Add Input'.")
    
    print(f"[*] Đang giải nén {os.path.basename(tar_path)} vào {WORKING_DIR} (khoảng 1-2 phút)...")
    !tar -xf "{tar_path}" -C {WORKING_DIR}
    DATA_ROOT = find_bhsd_root(WORKING_DIR)
    assert DATA_ROOT, f"[X] Giải nén xong nhưng không tìm thấy data/ và splits/ trong {WORKING_DIR}!"

print(f"[SUCCESS] Đường dẫn DATA_ROOT sẵn sàng: {DATA_ROOT}")

### Bước 4: Kiểm tra GPU & Tính toàn vẹn Dữ liệu

In [ ]:
import torch, os, glob

print("=" * 60)
assert torch.cuda.is_available(), "[CẢNH BÁO] Chưa bật GPU! Hãy chọn GPU T4 hoặc P100 ở Notebook Settings."
num_gpus = torch.cuda.device_count()
print(f"[GPU] Phát hiện {num_gpus} GPU khả dụng:")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)
    print(f"  [+] GPU {i}: {name} | VRAM: {vram} GB")
if num_gpus >= 2:
    print("[*] Đã sẵn sàng chế độ ĐA GPU (T4 x 2 - DataParallel) để tăng tốc và chia tải VRAM!")
else:
    print("[*] Chế độ đơn GPU (P100 hoặc 1x T4) đã sẵn sàng.")

train_split = os.path.join(DATA_ROOT, "splits", "fold0_train.txt")
val_split = os.path.join(DATA_ROOT, "splits", "fold0_val.txt")
labeled_split = os.path.join(DATA_ROOT, "train_labeled.txt")
if not os.path.exists(labeled_split):
    labeled_split = os.path.join(DATA_ROOT, "splits", "train_labeled.txt")
data_dir = os.path.join(DATA_ROOT, "data")

for p, name in [(train_split, "fold0_train.txt"), (val_split, "fold0_val.txt"), (data_dir, "data directory")]:
    assert os.path.exists(p), f"[LỖI] Thiếu {name} tại {p}"

with open(train_split) as f: train_cases = [line.strip() for line in f if line.strip()]
with open(val_split) as f: val_cases = [line.strip() for line in f if line.strip()]
labeled_count = len([line.strip() for line in open(labeled_split) if line.strip()]) if os.path.exists(labeled_split) else 0
h5_count = len(glob.glob(os.path.join(data_dir, "*.h5")))

print(f"[+] Số mẫu train:       {len(train_cases)} (Kỳ vọng: 2132)")
print(f"[+] Số mẫu val:         {len(val_cases)} (Kỳ vọng: 38)")
print(f"[+] Số mẫu có nhãn:     {labeled_count} (Kỳ vọng: 154)")
print(f"[+] Tổng file .h5 data: {h5_count} (Kỳ vọng: 2170)")
print("[OK] DỮ LIỆU ĐÃ ĐẦY ĐỦ VÀ SẴN SÀNG HUẤN LUYỆN!")
print("=" * 60)

### Bước 5: Đăng nhập & Cấu hình Weights & Biases (W&B)
Đăng nhập tài khoản W&B bằng API Key của bạn để truyền dữ liệu và vẽ biểu đồ trực tiếp lên Cloud.

In [ ]:
import wandb

# API Key W&B của bạn
WANDB_API_KEY = "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"
WANDB_PROJECT = "PICK_BHSD"

print("[*] Đang kết nối tới Weights & Biases...")
wandb.login(key=WANDB_API_KEY)
print(f"[SUCCESS] Đăng nhập W&B thành công! Dữ liệu huấn luyện sẽ được lưu tại project: '{WANDB_PROJECT}'")

### Bước 6: Huấn luyện PICK Multi-Class Segmentation (Tối ưu Đa GPU & An toàn <12h)
- **Tự động kích hoạt 2 GPU**: Cờ `--gpu auto` tự động sử dụng cả 2 GPU T4 (DataParallel) nếu có, giúp giảm tải VRAM mỗi card (~6-7GB) và rút ngắn thời gian train xuống **~6.5 - 7.5 tiếng**.
- **Tối ưu thời gian Validation**: Cờ `--val_interval 500` giúp giảm số lần đánh giá dư thừa, tiết kiệm hơn 3 tiếng mà vẫn theo dõi đường cong học tập chính xác.
- **Bảo vệ Checkpoint trên W&B Cloud**: Cờ `--save_wandb_model` tự động đẩy checkpoint tốt nhất (`best_model.pth`) lên Cloud W&B Artifacts theo thời gian thực. **Bạn không bao giờ lo mất checkpoint** dù Kaggle có ngắt hay timeout!
- **Cơ chế Auto-Resume**: Nếu phiên chạy bị gián đoạn, cờ `--resume` sẽ tự động tìm checkpoint gần nhất (`checkpoint_latest.pth`) để tiếp tục chính xác từ iteration đó.
- **Theo dõi thời gian thực & Bắt lỗi an toàn**: Stream trực tiếp tốc độ `s/it`, `ETA`, `VRAM`, tiến độ validation từng ca và bắt lỗi nếu tiến trình gặp sự cố.

In [ ]:
# === HUẤN LUYỆN MULTI-CLASS (FOLD 0) CÙNG W&B & TỐI ƯU 2 GPU (STREAM LOG & BẮT LỖI) ===
import os, sys, subprocess

FOLD_ID = 0
EXP_NAME = "BHSD_multiclass_PICK_fold0"
EXP_DIR = "/kaggle/working/experiments"

train_cmd = [
    "python", "-u", "BHSD_multiclass_PICK_train.py",
    "--root_path", str(DATA_ROOT),
    "--exp", EXP_NAME,
    "--exp_dir", EXP_DIR,
    "--fold", str(FOLD_ID),
    "--model", "VNet",
    "--labelnum", "154",
    "--max_samples", "2132",
    "--batch_size", "4",
    "--labeled_bs", "2",
    "--pre_max_iteration", "4000",
    "--self_max_iteration", "12000",
    "--val_interval", "500",
    "--gpu", "auto",
    "--phase", "all",
    "--resume",
    "--use_wandb",
    "--save_wandb_model",
    "--wandb_project", str(WANDB_PROJECT)
]

print("=" * 75)
print("[*] Khởi chạy huấn luyện mô hình PICK BHSD Multi-class (Fold 0)...")
print(f"[*] Theo dõi biểu đồ W&B Cloud tại: https://wandb.ai/{WANDB_PROJECT}")
print("[*] Logs chi tiết, tốc độ (s/it), ETA và VRAM hiển thị trực tiếp theo thời gian thực:")
print("=" * 75)

process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

process.wait()

print("\n" + "=" * 75)
if process.returncode != 0:
    raise RuntimeError(f"[LỖI NGHIÊM TRỌNG] Quá trình huấn luyện bị gián đoạn với mã lỗi {process.returncode}!")
else:
    print("[SUCCESS] Huấn luyện hoàn tất 100%! Đang chuyển sang Bước 7 nén kết quả...")
print("=" * 75)

### Bước 7: Nén và Xuất Kết quả Model Checkpoints & Logs
Nén toàn bộ kết quả mô hình (checkpoint tốt nhất `VNet_best_model.pth`, log TensorBoard, log `train.log`) thành file `.tar.gz` để tải về tại mục **Output**.
*(Lưu ý: Toàn bộ biểu đồ loss, dice và lịch sử huấn luyện đã được lưu trữ vĩnh viễn trên W&B Cloud)*.

In [ ]:
import os

OUTPUT_ARCHIVE = "/kaggle/working/bhsd_multiclass_fold0_results.tar.gz"
EXP_DIR = "/kaggle/working/experiments"

if os.path.exists(EXP_DIR):
    print("[*] Đang nén toàn bộ kết quả mô hình Multi-class và logs...")
    !tar -czvf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[OK] Đã nén thành công file kết quả: {OUTPUT_ARCHIVE} ({size_mb} MB)")
    print("[*] File đã sẵn sàng tải về tại tab Output bên phải màn hình!")
    print("[*] Bạn có thể vào https://wandb.ai/ để xem lại toàn bộ biểu đồ và phân tích so sánh.")
else:
    print(f"[!] Chưa tìm thấy thư mục {EXP_DIR}. Vui lòng kiểm tra lại quá trình huấn luyện.")